In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

os.makedirs("data/processed", exist_ok=True)

# 1. Load the merged raw CSV
df = pd.read_csv("../data/processed/cp_data.csv", keep_default_na=False)

for col in ["xpotime", "xprtime", "ndoses", "max_dose", "min_dose", "mean_dose", "num_dose_groups"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 2. Group tissue-tumor rows into 1 row per experiment
exp_keys = ["chemcode", "papernum", "species", "strain", "sex", "route"]

def get_experiment_target(opinions):
    ops = set(opinions)
    if "+" in ops:
        return 1       # Carcinogenic
    elif "-" in ops:
        return 0       # Non-carcinogenic
    return -1          # 0-only opinions (saved separately)

experiments = (
    df.groupby(exp_keys, as_index=False)
    .agg(
        name=("name", "first"),
        cas=("cas", "first"),
        xpotime=("xpotime", "max"),
        xprtime=("xprtime", "max"),
        ndoses=("ndoses", "max"),
        max_dose=("max_dose", "max"),
        min_dose=("min_dose", "min"),
        mean_dose=("mean_dose", "mean"),
        num_dose_groups=("num_dose_groups", "max"),
        is_carcinogenic=("opinion", get_experiment_target),
    )
)

experiments["log_max_dose"] = np.log10(experiments["max_dose"])

# Save 0-only experiments separately
experiments[experiments["is_carcinogenic"] == -1].to_csv(
    "../data/training/experiments_zero_only.csv", index=False
)

# Keep binary (+ vs -) experiments for modeling
clean_df = experiments[experiments["is_carcinogenic"].isin([0, 1])].reset_index(drop=True)
clean_df.to_csv("../data/training/experiments.csv", index=False)

# 3. Split by unique chemical using standard train_test_split
unique_chems = clean_df["chemcode"].unique()
train_chems, test_chems = train_test_split(unique_chems, test_size=0.2, random_state=42)

feature_cols = [
    "species", "strain", "sex", "route",
    "xpotime", "xprtime", "ndoses",
    "log_max_dose", "num_dose_groups",
]

X_all = pd.get_dummies(clean_df[feature_cols], drop_first=True)
y_all = clean_df["is_carcinogenic"]

X_train = X_all[clean_df["chemcode"].isin(train_chems)]
X_test = X_all[clean_df["chemcode"].isin(test_chems)]
y_train = y_all[clean_df["chemcode"].isin(train_chems)]
y_test = y_all[clean_df["chemcode"].isin(test_chems)]

X_train.to_csv("../data/training/X_train.csv", index=False)
X_test.to_csv("../data/training/X_test.csv", index=False)
y_train.to_csv("../data/training/y_train.csv", index=False)
y_test.to_csv("../data/training/y_test.csv", index=False)

print(f"Step 2 complete! {len(clean_df)} experiments -> {len(X_train)} train, {len(X_test)} test.")

Step 2 complete! 3947 experiments -> 3178 train, 769 test.
